# 03 - Training Baseline

This notebook trains and evaluates the baseline model by calling the existing scripts with `subprocess`. It uses normal Python code only for lightweight checks and file copying.

## Step 1 - Setup

In [ ]:
from pathlib import Path
import shutil
import sys

def find_notebooks_dir():
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd / "notebooks", cwd / "salient-object-detection-cnn" / "notebooks"]
    for parent in cwd.parents:
        candidates.extend([parent / "notebooks", parent / "salient-object-detection-cnn" / "notebooks"])
    for candidate in candidates:
        if (candidate / "notebook_utils.py").exists():
            return candidate
    raise FileNotFoundError("Could not find notebooks/notebook_utils.py")

NOTEBOOKS_DIR = find_notebooks_dir()
sys.path.insert(0, str(NOTEBOOKS_DIR))

from notebook_utils import PROJECT_DIR, run_script, run_script_live

sys.path.insert(0, str(PROJECT_DIR))
print(f"Using project directory: {PROJECT_DIR}")

## Step 2 - Configure Baseline Run

In [ ]:
IMAGE_SIZE = 128
BATCH_SIZE = 16
EPOCHS = 25
LR = 1e-3
MODEL_TYPE = "baseline"
NUM_WORKERS = 2

## Step 3 - Confirm Preprocessed Data Exists

In [ ]:
manifest_path = PROJECT_DIR / "pre-processed/manifest.json"
if not manifest_path.exists():
    raise FileNotFoundError("Missing pre-processed/manifest.json. Run 01_data_preprocessing.ipynb first.")
print("Found:", manifest_path)

## Step 4 - Train Baseline Model

In [ ]:
run_script_live(
    "train.py",
    "--data_dir", "pre-processed",
    "--image_size", IMAGE_SIZE,
    "--batch_size", BATCH_SIZE,
    "--epochs", EPOCHS,
    "--lr", LR,
    "--model_type", MODEL_TYPE,
    "--num_workers", NUM_WORKERS,
 )

## Step 5 - Evaluate Baseline On Test Split

In [ ]:
run_script_live(
    "evaluate.py",
    "--data_dir", "pre-processed",
    "--image_size", IMAGE_SIZE,
    "--batch_size", BATCH_SIZE,
    "--model_type", MODEL_TYPE,
    "--checkpoint", "checkpoints/best_model_baseline.pth",
    "--num_workers", NUM_WORKERS,
)

source = PROJECT_DIR / "outputs/metrics/test_metrics.json"
target = PROJECT_DIR / "outputs/metrics/test_metrics_baseline.json"
if source.exists():
    shutil.copyfile(source, target)
    print(f"Copied {source} to {target}")

## Step 6 - Generate Baseline Visualizations

In [ ]:
run_script_live(
    "visualize.py",
    "--data_dir", "pre-processed",
    "--image_size", IMAGE_SIZE,
    "--model_type", MODEL_TYPE,
    "--checkpoint", "checkpoints/best_model_baseline.pth",
    "--num_samples", 10,
)